# Plexgraph evaluation lab

This is a validation notebook, separate from the feature tour. A rendered graph is not evidence of a useful layout. Measure layout quality, resource use, interaction latency, and whether a person can answer a graph question.

**Current implementation:** bounded exact/mesh repulsion, centroid hyperedge attraction, analytic paths/cycles, and component packing replace the failing baseline. Large viewers use labeled density overviews; search reveals actual neighborhoods. Compare historical and fixed results in `benchmarks/results/`. A production-scale claim still needs target-hardware and real-data validation.

Run the quick suite first. Stress tests and viewers are opt-in so **Run All** does not launch a 100K-node browser session. Workers have time and memory limits. The process resource limiter currently requires Linux/macOS; reported timing and memory depend on the machine.

## 1. Setup

**Environment.** Run this notebook with the repo's Python environment, which has every dependency (NumPy, msgpack, websockets, the optional pandas/networkx loaders, and the notebook and benchmark tools): `uv sync`, or `python -m venv .venv && .venv/bin/pip install -r requirements-dev.txt`, then pick the `.venv` interpreter as the notebook kernel. The `sys.path` lines below only make the local packages importable; they do not install dependencies, so a kernel from a different environment fails at `import numpy`.

Interactive viewers additionally need a built frontend (`pnpm --filter @plexgraph/app build`), and the browser checks need Node with Playwright. Results include source revision and dirty-tree status; keep the tested patch with the report. Cached results in `benchmarks/results/quality/` are shown by default; set the `RERUN_*` flags to regenerate them.


In [ ]:
import sys
from pathlib import Path

repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "packages/core/plexgraph_core").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Start Jupyter inside the plexgraph checkout")
for path in (repo_root, repo_root / "packages/core", repo_root / "packages/bridge"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import json
from benchmarks.layout_audit import build_case, run_case, run_suite
from plexgraph_core.algorithms.layout import force_directed_layout
import numpy as np


## 2. Layout quality against established engines

The audit below asks *is the layout good?*, not *does it run?* Every engine lays out the same graph with the same iteration budget (100) and seed. `random` is the floor; `igraph-fr/drl/kk` and `networkx-spring` are the established baselines.

| Metric | Better | Meaning |
|---|---|---|
| `stress` | lower | Normalised gap between layout distance and graph distance (sampled sources). |
| `neighbourhood_precision` | higher | Share of a node's graph neighbours among its *k* nearest layout neighbours. Chance is about k/n. |
| `community_silhouette` | higher | Separation of ground-truth communities in the layout (only where labels exist). |
| `edge_length_cv` | lower | Uniformity of edge lengths. |
| `crossings_per_edge_pair` | lower | Crossing density of 1,500 sampled non-adjacent edges. |
| `overlap_fraction` | lower | Nodes whose nearest neighbour is closer than a 6px marker after fitting a 1000x700 viewport. |

Datasets: Zachary karate club and Les Miserables (tiny, real), a balanced tree and a grid (regular structure with a known answer), stochastic block models with planted communities at 1K/5K/20K, a Barabasi-Albert scale-free graph, and the SNAP Facebook ego network (4,039 nodes, 88,234 edges, downloaded on first use).

In [ ]:
from IPython.display import Markdown, Image, display

QUALITY_DIR = repo_root / "benchmarks/results/quality"
RERUN_QUALITY = False   # about 4 minutes; downloads the SNAP Facebook graph on first use
if RERUN_QUALITY:
    import subprocess
    subprocess.run([sys.executable, str(repo_root / "benchmarks/quality_report.py"), "--out", str(QUALITY_DIR)], check=True)
quality = json.loads((QUALITY_DIR / "quality.json").read_text())
display(Markdown((QUALITY_DIR / "quality.md").read_text()))

### 2.1 Automatic verdicts

plexgraph is compared with the **median of the established engines** on each metric. It fails a metric if it is more than 25% worse *and* the absolute gap is meaningful (0.02, or 1 second for time). This is a regression tripwire, not a proof of quality.

In [ ]:
LOWER = {"stress", "edge_length_cv", "crossings_per_edge_pair", "overlap_fraction", "seconds"}
HIGHER = {"neighbourhood_precision", "community_silhouette"}
TOL = 1.25
ORDER = ["stress", "neighbourhood_precision", "community_silhouette", "edge_length_cv",
         "crossings_per_edge_pair", "overlap_fraction", "seconds"]

def verdicts(rows):
    out = {}
    for ds in dict.fromkeys(r["dataset"] for r in rows):
        mine = next((r for r in rows if r["dataset"] == ds and r["engine"] == "plexgraph" and r["status"] == "ok"), None)
        refs = [r for r in rows if r["dataset"] == ds and r["engine"] not in ("plexgraph", "random") and r["status"] == "ok"]
        for m in ORDER:
            vals = [r[m] for r in refs if isinstance(r.get(m), float)]
            if mine is None or not vals or not isinstance(mine.get(m), float):
                continue
            ref, v = float(np.median(vals)), mine[m]
            floor = 1.0 if m == "seconds" else 0.02
            bad = (v > ref * TOL and v - ref > floor) if m in LOWER else (v < ref / TOL and ref - v > floor)
            out[(ds, m)] = (v, ref, bad)
    return out

V = verdicts(quality)
datasets = list(dict.fromkeys(ds for ds, _ in V))
print(f"{'dataset':10}" + "".join(f"{m[:12]:>15}" for m in ORDER))
for ds in datasets:
    cells = []
    for m in ORDER:
        v = V.get((ds, m))
        cells.append("-" if v is None else f"{'FAIL' if v[2] else 'ok  '} {v[0]:.2f}/{v[1]:.2f}")
    print(f"{ds:10}" + "".join(f"{c:>15}" for c in cells))
layout_failures = [(ds, m) for (ds, m), v in V.items() if v[2]]
print(f"\n{len(layout_failures)} of {len(V)} checks worse than the established-engine median (value/reference shown)")

### 2.2 Look at the layouts

Numbers can hide problems that are obvious by eye. Each strip shows the same graph laid out by each engine. Colour is the planted community where one exists, otherwise log-degree.

In [ ]:
for png in sorted(QUALITY_DIR.glob("gallery-*.png")):
    display(Image(filename=str(png), width=1150))

## 3. The real viewer, on real datasets

Section 2 judges coordinates. This section opens the *shipped viewer* through the real bridge and Chrome (software GL), screenshots it, counts what it draws, and records console errors. Two automatic checks:

- **Canvas opacity**: the page background is opaque, so a correctly blended canvas has alpha 255 everywhere. Translucent pixels holding non-premultiplied colours composite brighter than intended, which shows up as white edges cutting through nodes.
- **Clean console**: no page errors or failed requests.
- **Restyle**: applying a color-by-degree, size-by-degree and outline style must finish quickly (under 3 seconds) at every size.
- **Drill-down**: above 5,000 nodes the viewer aggregates, so zooming in must reach individual nodes in the current view; the check fails if it stays aggregated.

In [ ]:
VIEWER_DATASETS = "karate,sbm1000,sbm5000,facebook,sbm20000"
RERUN_VIEWER = False   # a few minutes; needs `pnpm --filter @plexgraph/app build` and Playwright's Chrome
if RERUN_VIEWER:
    import os, subprocess
    subprocess.run(["node", "scripts/viewer-gallery.mjs", f"--datasets={VIEWER_DATASETS}", f"--out={QUALITY_DIR}"],
                   cwd=repo_root, check=True, env=dict(os.environ, PYTHON_PATH=sys.executable))
gallery = json.loads((QUALITY_DIR / "viewer-gallery.json").read_text())
viewer_failures = []
print(f"{'dataset':10}{'nodes':>8}{'overview':>10}{'zoomed':>9}{'restyle ms':>12}{'translucent px':>16}  result")
for row in gallery:
    px = row.get("pixels", {})
    bad = []
    if px.get("translucentPixels", 0) > 0: bad.append("translucent canvas")
    if row.get("errors"): bad.append("console errors")
    if row.get("styleMs", 0) > 3000:
        bad.append(f"restyling took {row['styleMs']} ms")
    zoomed = row.get("lod", {}).get("zoomed")
    if row.get("nodes", 0) > 5000 and (zoomed is None or zoomed["mode"] not in ("region", "detail")):
        bad.append("zooming in never reaches individual nodes")
    if row["status"] != "ok": bad.append(row.get("error", "failed"))
    viewer_failures += [(row["dataset"], b) for b in bad]
    lod = row.get("lod", {})
    print(f"{row['dataset']:10}{row.get('nodes', 0):>8}{lod.get('initial', {}).get('mode', '-'):>10}"
          f"{(lod.get('zoomed') or {}).get('mode', '-'):>9}{row.get('styleMs', '-'):>12}{px.get('translucentPixels', 0):>16}  {'; '.join(bad) or 'ok'}")
for row in gallery:
    display(Markdown(f"**{row['dataset']}**  (overview, restyled, zoomed in where the graph is aggregated, one node's neighbourhood)"))
    for suffix in ("", "-styled", "-zoom", "-focus"):
        f = QUALITY_DIR / f"viewer-{row['dataset']}{suffix}.png"
        if f.exists():
            display(Image(filename=str(f), width=420))

## 4. Capability checklist

What can a person *do* with the viewer? This inventories the live UI (buttons, inputs) and the public `ViewerHandle` methods captured in section 3, and checks them against the capabilities needed to explore a large network. A missing capability is a result, not an omission from the list.

In [ ]:
controls = gallery[-1]["controls"]
methods = {m.lower() for m in controls["methods"]}
labels = " | ".join(controls["buttons"] + controls["inputs"]).lower()
has_method = lambda *words: any(all(w in m for w in words) for m in methods)
CAPABILITIES = [
    ("Find a node by key",                          "search" in labels),
    ("Fit / reset view",                            "fit view" in labels),
    ("Neighbourhood focus and node inspector",      has_method("focusneighborhood") and has_method("inspectnode")),
    ("Layer filter",                                has_method("layerfilter")),
    ("Time filter",                                 has_method("timefilter")),
    ("Export image / vector / PDF",                 "svg vector" in labels and "png image" in labels),
    ("Filter by node attribute value",              has_method("setnodefilter")),
    ("Filter by degree or centrality threshold",    has_method("setnodefilter") and "minimum degree" in labels),
    ("Choose colour / size attribute from the UI",  "color by" in labels and "size by" in labels),
    ("Choose or re-run the layout algorithm",       has_method("layout") or "layout algorithm" in labels),
    ("Multi-node selection",                        has_method("highlightednodes") and has_method("getnodescreenposition")),
    ("Shortest path between two nodes",             has_method("path")),
    ("Load temporal (u, v, t) contact sequences",   all(hasattr(__import__("plexgraph_core"), n) for n in ("from_temporal_edgelist", "read_temporal_edgelist", "from_pandas_temporal_edgelist"))),
    ("Set node and edge size from the viewer",     "node size" in labels and "edge width" in labels),
    ("Color by attribute, degree, weight, time or time bucket", "node color mode" in labels and "edge color mode" in labels),
    ("Recolor chosen nodes (paint) live",            has_method("paintnodes") and "paint selected nodes" in labels),
    ("Node shapes, outlines and curved edges",       has_method("setstyle") and "node shape" in labels and "edge curvature" in labels),
    ("Restyle an open viewer from Python (networkx-style)", all(hasattr(__import__("plexgraph_bridge"), n) for n in ("by_attribute", "by_time_bucket", "size_by_degree")) and hasattr(__import__("plexgraph_bridge.launcher", fromlist=["ShowHandle"]).ShowHandle, "style")),
    ("Collapse / expand communities",               has_method("setgroupby")),
    ("Pin or drag nodes",                           has_method("pin") or has_method("drag")),
    ("Zoom controls or minimap",                    "zoom in" in labels and "zoom out" in labels),
]
for name, present in CAPABILITIES:
    print(f"{'PRESENT' if present else 'MISSING':8}{name}")
missing = [n for n, ok in CAPABILITIES if not ok]
print(f"\n{len(missing)} of {len(CAPABILITIES)} capabilities missing")
print("Public ViewerHandle methods:", ", ".join(sorted(controls["methods"])))

## 5. Structural cases, three seeds each

Paths test ordering; stars test hub crowding; planted communities test separation; disconnected graphs and isolates test packing. Multiplex and temporal cases test the aggregate layout. Large hyperedges test pair expansion.

`crowded_cell_fraction` is an 8px grid occupancy proxy after fitting a 1000×700 viewport, **not** an exact overlap count. `edge_to_random_distance` compares median edge length with sampled random-pair distance; a low ratio can coexist with unreadable clusters. Neither is a standalone quality score. `termination_flag` reports the implementation's stopping condition, not proven convergence.

In [ ]:
quick = run_suite("quick", timeout=20, memory_mb=1024)
output = Path("/tmp/plexgraph-quick-audit.json")
output.write_text(json.dumps(quick, indent=2))
for row in quick["results"]:
    print(row["kind"], row["seed"], row["status"],
          "seconds=", round(row.get("layout_seconds", 0), 3),
          "crowding=", round(row.get("crowded_cell_fraction", 0), 3),
          "edge/random=", row.get("edge_to_random_distance"))
assert all(r["status"] == "ok" and r["finite"] for r in quick["results"])


## 6. Reproducibility and immutability

These are correctness checks, not quality checks. Stream results without keeping every coordinate array for large graphs.

In [ ]:
g = build_case("communities", 120, 0)
def final_layout(g, seed):
    last = None
    for last in force_directed_layout(g, seed=seed, iterations=60):
        assert np.isfinite(last.positions).all()
    return last.positions
before = [(c.id, c.endpoints) for c in g.connectors()]
a = final_layout(g, 7)
b = final_layout(g, 7)
np.testing.assert_array_equal(a, b)
assert before == [(c.id, c.endpoints) for c in g.connectors()]
print("Deterministic coordinates; graph topology unchanged")


## 7. Inspect topology, not decoration

Optional static plots use matplotlib. Compare multiple seeds and record failures: path folding, hub congestion, mixed communities, overlapping disconnected components, and isolated-node drift. Community color is ground truth from the fixture, not a detected result.

In [ ]:
PLOT_LAYOUTS = False  # requires matplotlib; no dependency for numerical audit
if PLOT_LAYOUTS:
    import matplotlib.pyplot as plt
    from matplotlib.collections import LineCollection
    fig, axes = plt.subplots(2, 3, figsize=(14, 9))
    for ax, kind in zip(axes.flat, ["path", "star", "communities", "disconnected", "isolates", "hyperedge"]):
        graph = build_case(kind, 120, 0)
        p = final_layout(graph, 0)
        edges = [p[list(c.endpoints)] for c in graph.connectors() if len(c.endpoints) == 2]
        ax.add_collection(LineCollection(edges, colors="#64748b", linewidths=.4, alpha=.35))
        ax.scatter(p[:, 0], p[:, 1], c=np.arange(120) // 30, s=10, cmap="tab10")
        ax.set_title(kind); ax.set_aspect("equal"); ax.autoscale_view()
    fig.tight_layout()


## 8. Threshold and stress audit — explicit opt-in

Test 1K, 3K, 5K, 5,001, 10K, and 100K nodes. The adjacent 5K/5,001 cases guard against the former algorithm discontinuity. Hyperedge cases guard against quadratic attraction expansion. A timeout or memory exception is a result, not a reason to remove the fixture. Limits apply per worker; no result arrays accumulate across iterations.

A successful Python result says nothing about browser frame time, memory, or task success.

In [ ]:
RUN_STRESS = False
if RUN_STRESS:
    scale = run_suite("scale", timeout=30, memory_mb=1024)
    Path("/tmp/plexgraph-scale-audit.json").write_text(json.dumps(scale, indent=2))
    for row in scale["results"]:
        print(row["kind"], row["nodes"], row["status"],
              row.get("first_step_seconds"), row.get("peak_rss_mb"))


## 9. Interactive task protocol

Open one viewer at a time. Use `show(..., return_handle=True)` and call `handle.close()` before replacing a session. Closing is idempotent and releases the servers.

For each task record graph size, seed, view, elapsed task time, correct/incorrect answer, and the missing control:

| Task | Required capability | Current gap |
|---|---|---|
| Find an entity by key | Search and focus | Search and result focus implemented |
| Explain a hub's connections | Neighborhood isolation and attributes | Incident-neighborhood focus and attributes implemented |
| Compare a node across layers | Linked selection and filtering | Focused identity persists across slices |
| Find when a relationship changes | Time selection and differences | Interval summaries; no difference view |
| Inspect 100K nodes | Aggregation, level of detail, bounded picking | Density overview; detailed picking within focused neighborhoods |
| Return to a result | Saved view and selection state | Not available |

Do not mark a task passed just because the graph is visible.

In [ ]:
OPEN_VIEWER = False
if OPEN_VIEWER:
    if "handle" in globals() and handle is not None:
        handle.close()
    from plexgraph_bridge import show
    graph = build_case("multiplex", 500, 0)
    handle = show(graph, seed=0, node_color_by="community", layout_iterations=60, return_handle=True)


## 10. Browser benchmark — separate from Python layout

Start `pnpm --filter @plexgraph/app dev --host 127.0.0.1`, then run:

```sh
node scripts/benchmark-render.mjs --sizes=1000,10000 --output=/tmp/plexgraph-browser.json
# Explicit stress run:
node scripts/benchmark-render.mjs --sizes=100000 --output=/tmp/plexgraph-browser-100k.json
```

The benchmark uses deterministic synthetic positions to isolate renderer cost. It measures graph upload, layout upload, slice construction, frame intervals, and hover work; it does not measure layout quality, bridge transport, or end-user task success. Software-rendered Chrome results are a regression baseline, not a hardware GPU claim.

Also run `node scripts/verify-slices.mjs` for small-graph arrow/hull/export/hover correctness. Do not generate full SVG exports of 100K-node sliced graphs as a default smoke check.

## 11. Release gates to agree and measure

- Correctness: deterministic seeds, finite positions, graph immutability, filters and direction preserved, export parity.
- Layout: review structured graphs against established baselines and record task accuracy; compare the mesh approximation and component packing against external layout baselines before advertising large-graph layout quality.
- Interaction: verify search, neighborhood focus, inspector, linked focus, and fit/reset; add meaningful temporal difference views.
- Scale: record p50/p95 frame intervals, hover latency, time to first frame, peak memory, and failures on target hardware, for both overview and slices.
- Lifecycle: session disposal, reconnect/disconnect, repeated loads, resizing, and memory recovery.

Proposed interaction budgets for discussion: p95 frame interval ≤33ms and hover feedback ≤100ms at the chosen supported workload. These are acceptance targets, not achieved results. Large networks use aggregation. Record that representation in every performance comparison; fast population marks do not establish individual-node readability.

## 12. A real temporal network: Reddit hyperlinks

The SNAP *Reddit Hyperlink Network* (title version, `soc-redditHyperlinks-title.tsv`, 368 MB) is a directed, timestamped edge list: one row per hyperlink from one subreddit to another, with text timestamps. It is not shipped with the repo; put it in `sample_data/` to run this section. It exercises the loaders and every temporal view on real data: text timestamps, about 54K nodes, about 572K events, repeated pairs, and node names that look like numbers (one subreddit is called `24811812513198111524`, which is larger than any 64-bit integer).

Checks: the file loads with the right shape; timestamps became dates; the viewer's window filter selects exactly the events a plain NumPy comparison does; and the whole graph fits through the wire protocol.

In [ ]:
import time
import datetime as dt

REDDIT = repo_root / "sample_data/soc-redditHyperlinks-title.tsv"
reddit_failures = []
if not REDDIT.exists():
    print("sample_data/soc-redditHyperlinks-title.tsv not found; skipping this section")
else:
    from plexgraph_core import read_temporal_edgelist
    from plexgraph_core.wire.protocol import decode, encode_graph
    started = time.perf_counter()
    reddit = read_temporal_edgelist(REDDIT, delimiter="\t", header=True, columns=(0, 1, 3),
                                    attrs={"sentiment": 4}, directed=True)
    load_seconds = time.perf_counter() - started
    starts = np.array([c.t_start for c in reddit.connectors()])
    first, last = (dt.datetime.fromtimestamp(x, dt.timezone.utc) for x in (starts.min(), starts.max()))
    print(f"loaded in {load_seconds:.1f}s: {reddit.num_nodes:,} nodes, {len(starts):,} events, "
          f"{first:%Y-%m-%d} to {last:%Y-%m-%d}, time_unit={reddit.time_unit}")

    # The viewer's trailing window is start <= t and end >= t - window; recompute it independently.
    t, window = starts.min() + (starts.max() - starts.min()) / 2, (starts.max() - starts.min()) / 20
    expected = int(((starts >= t - window) & (starts <= t)).sum())
    print(f"events in a trailing window of {window / 86400:.0f} days ending {dt.datetime.fromtimestamp(t, dt.timezone.utc):%Y-%m-%d}: {expected:,}")
    print(f"events at exactly that instant: {int((starts == t).sum())}  (why point-event data needs a window view)")

    started = time.perf_counter()
    wire = encode_graph(reddit)
    message = decode(wire)
    print(f"wire message {len(wire) / 1e6:.0f} MB in {time.perf_counter() - started:.1f}s; "
          f"big node name survives as {[n['key'] for n in message['nodes'] if str(n['key']).startswith('2481181251')]}")

    if reddit.time_unit != "epoch_seconds": reddit_failures.append("timestamps were not recognised as dates")
    if len(starts) != 571_927 or reddit.num_nodes < 50_000: reddit_failures.append("unexpected size")
    if load_seconds > 30: reddit_failures.append(f"load took {load_seconds:.0f}s")
    print("failures:", reddit_failures or "none")

In [ ]:
OPEN_REDDIT_VIEWER = False   # opens the viewer on the full graph; scrub the timeline (default: trailing window)
if OPEN_REDDIT_VIEWER and REDDIT.exists():
    from plexgraph_bridge import show
    if "reddit_handle" in globals() and reddit_handle is not None:
        reddit_handle.close()
    reddit_handle = show(reddit, seed=0, layout_iterations=30, return_handle=True)
    print(reddit_handle.url)

## 13. Scorecard

A single summary of the automatic checks above. It counts only what is measured here; unlisted things (real-hardware GPU speed, user task studies) are still unmeasured.

In [ ]:
print(f"Layout quality vs established engines : {len(layout_failures)} failing metric(s) of {len(V)}")
print(f"Viewer defects (canvas / console)     : {len(viewer_failures)}")
for item in viewer_failures: print("   ", *item)
print(f"Real temporal dataset (Reddit)      : {len(reddit_failures)} failure(s)" if REDDIT.exists() else "Real temporal dataset (Reddit)      : skipped (file not present)")
for item in reddit_failures: print("   ", item)
print(f"Missing exploration capabilities      : {len(missing)} of {len(CAPABILITIES)}")
for item in missing: print("    ", item)